# Experiment 7: Simple Autoencoder for Dimensionality Reduction and Feature Learning
**Name:** Fereena Mary Catherine  
**Roll No:** 24BAD024  
**Dataset:** MNIST handwritten digits

## Part A – Dataset Preparation

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import mnist

print("TensorFlow version:", tf.__version__)

# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print("Training data shape:", x_train.shape)
print("Testing data shape:", x_test.shape)
print("Data type:", x_train.dtype)

# Normalize pixel values from 0-255 to 0-1
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Flatten 28x28 images into 784 features
x_train_flat = x_train.reshape(-1, 784)
x_test_flat = x_test.reshape(-1, 784)

print("Flattened training shape:", x_train_flat.shape)
print("Flattened testing shape:", x_test_flat.shape)

# Display sample original images
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.axis("off")
    plt.title(str(y_train[i]))
plt.suptitle("Original MNIST Images")
plt.show()

## Part B – Implementing the Autoencoder Model

In [ ]:
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import Dense

input_dim = 784
latent_dim = 32

# Encoder: 784 -> 128 -> 64 -> 32
input_layer = Input(shape=(input_dim,), name="input")
encoded = Dense(128, activation="relu")(input_layer)
encoded = Dense(64, activation="relu")(encoded)
latent = Dense(latent_dim, activation="relu", name="bottleneck")(encoded)

# Decoder: 32 -> 64 -> 128 -> 784
decoded = Dense(64, activation="relu")(latent)
decoded = Dense(128, activation="relu")(decoded)
output_layer = Dense(input_dim, activation="sigmoid")(decoded)

# Complete autoencoder
autoencoder = Model(input_layer, output_layer, name="Autoencoder")

# Separate encoder and decoder models
encoder = Model(input_layer, latent, name="Encoder")
latent_input = Input(shape=(latent_dim,))
decoder_layers = autoencoder.layers[-3](latent_input)
decoder_layers = autoencoder.layers[-2](decoder_layers)
decoder_output = autoencoder.layers[-1](decoder_layers)
decoder = Model(latent_input, decoder_output, name="Decoder")

autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()
print("Encoder output shape:", encoder.output_shape)
print("Decoder output shape:", decoder.output_shape)

## Part C – Model Training and Performance Evaluation

In [ ]:
# Train the model; input and target are the same for reconstruction
history = autoencoder.fit(
    x_train_flat, x_train_flat,
    epochs=10,
    batch_size=256,
    shuffle=True,
    validation_data=(x_test_flat, x_test_flat),
    verbose=1
)

# Evaluate on test data
test_loss = autoencoder.evaluate(x_test_flat, x_test_flat, verbose=0)
print("Test Reconstruction MSE:", test_loss)

# Plot training and validation loss
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

# Compare original and reconstructed images
reconstructed = autoencoder.predict(x_test_flat[:10])
plt.figure(figsize=(12, 4))
for i in range(10):
    plt.subplot(2, 10, i + 1)
    plt.imshow(x_test[i], cmap="gray")
    plt.axis("off")
    if i == 0: plt.ylabel("Original")
    plt.subplot(2, 10, i + 11)
    plt.imshow(reconstructed[i].reshape(28, 28), cmap="gray")
    plt.axis("off")
    if i == 0: plt.ylabel("Reconstructed")
plt.suptitle("Original vs Reconstructed Images")
plt.show()

## Part D – Dimensionality Reduction and Feature Visualization

In [ ]:
# Extract compressed representations using the Encoder
latent_features = encoder.predict(x_test_flat)
print("Original test data shape:", x_test_flat.shape)
print("Compressed representation shape:", latent_features.shape)
print("Original dimensions:", x_test_flat.shape[1])
print("Latent dimensions:", latent_features.shape[1])

# Visualize latent features using first two latent dimensions
plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    latent_features[:2000, 0], latent_features[:2000, 1],
    c=y_test[:2000], cmap="tab10", s=8
)
plt.colorbar(scatter, label="Digit class")
plt.xlabel("Latent Feature 1")
plt.ylabel("Latent Feature 2")
plt.title("2D View of Autoencoder Latent Representations")
plt.show()

# Reconstruct selected test samples using the Decoder
compressed_samples = encoder.predict(x_test_flat[:5])
decoded_samples = decoder.predict(compressed_samples)

plt.figure(figsize=(10, 4))
for i in range(5):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i], cmap="gray")
    plt.title("Original")
    plt.axis("off")
    plt.subplot(2, 5, i + 6)
    plt.imshow(decoded_samples[i].reshape(28, 28), cmap="gray")
    plt.title("Reconstructed")
    plt.axis("off")
plt.tight_layout()
plt.show()

## Observation and Result
**Observation:** The original input contains 784 features, while the Encoder compresses each image into 32 latent features. The Decoder reconstructs images from these compressed features. The loss graph shows how reconstruction error changes during training.

**Result:** A Dense Autoencoder was built and trained on MNIST. Dimensionality reduction and feature extraction were performed, and reconstruction quality was evaluated using test MSE.

## Post-Lab Questions
1. Reconstruction loss measures the difference between original and reconstructed data and guides the model during training.
2. Autoencoders can learn complex non-linear patterns.
3. Applications include anomaly detection and image denoising.